# Project: How much data do I need? 📈
The most common real-world question in ML, answered with **learning curves**: train on 50, 100, 200 … examples and plot training and validation error. The shape tells you whether to collect more data (variance) or change the model (bias), before you spend a month labelling.

Topic: [[Bias-Variance Tradeoff]], [[Overfitting and Regularization]] · guide note: [[Project - How Much Data Do I Need]]

**How this works:** cells marked **TODO** are yours. Run the ✅ check under each one; later cells stay skipped until the checks they need pass. Stuck? Open the hidden ▶ *Solution*. Data is downloaded once into `./data` (git-ignored).

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from sklearn.base import clone
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
DONE = set()
def check(name, test, hint):
    """Runs test(); ✅ if it returns True. Passed checks unlock the cells further down."""
    try:
        ok = bool(test())
    except NotImplementedError:
        print(f"❌ {name}: not implemented yet – fill in the TODO above."); return
    except Exception as e:
        print(f"❌ {name}: {type(e).__name__}: {e}"); return
    if ok: DONE.add(name); print(f"✅ {name}")
    else: print(f"❌ {name}: {hint}")

def ready(*names):
    missing = [n for n in names if n not in DONE]
    if missing: print("⏭  Skipped – pass these checks first:", ", ".join(missing))
    return not missing
X, y = load_digits(return_X_y=True); X = X / 16
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.3, stratify=y, random_state=0)
SIZES = [30, 60, 120, 250, 500, 1000]

## 1. Learning curve
For each size `n`: sample `n` training examples (without replacement), fit a **fresh copy** of the model (`clone`), record training error on those `n` and validation error on the fixed validation set. Average over `repeats` samples. Return a list of `(n, train_err, val_err)`.

In [ ]:
def learning_curve(model, X_train, y_train, X_val, y_val, sizes, repeats=3, seed=0):
    # TODO 1: average train/val error (1 − accuracy) per training-set size
    raise NotImplementedError  # TODO 1

In [ ]:
def _t_lc():
    lc = learning_curve(DecisionTreeClassifier(random_state=0), X_train, y_train, X_val, y_val, [50, 400], repeats=2)
    return len(lc) == 2 and lc[0][0] == 50 and lc[0][1] == 0.0 and lc[1][2] < lc[0][2]
check("learning curve", _t_lc, "An unpruned tree has 0 training error; validation error must drop from 50 to 400 examples.")
curves = {}
if ready("learning curve"):
    models = {"logistic regression": LogisticRegression(max_iter=2000), "decision tree (depth 3)": DecisionTreeClassifier(max_depth=3, random_state=0), "RBF SVM": SVC()}
    fig, ax = plt.subplots(1, 3, figsize=(15, 3.6), sharey=True)
    for a, (name, m) in zip(ax, models.items()):
        lc = learning_curve(m, X_train, y_train, X_val, y_val, SIZES); curves[name] = lc
        n, tr, va = zip(*lc); a.plot(n, tr, "o-", label="train"); a.plot(n, va, "o-", label="validation"); a.set_xscale("log"); a.set_title(name); a.set_xlabel("training examples")
    ax[0].set_ylabel("error"); ax[0].legend(); plt.show()
    for name, lc in curves.items(): print(f"{name:<24} at n=1000: train {lc[-1][1]:.3f}  val {lc[-1][2]:.3f}")

## 2. Diagnose
Turn the curve into advice. Given the errors at the largest size and a target error: if validation error already meets the target → `"good enough"`; else if the train–validation gap is above 0.05 → `"high variance: …"`; else → `"high bias: …"` (start the strings with exactly those words).

In [ ]:
def diagnose(train_err, val_err, target=0.03):
    # TODO 2: good enough / high variance / high bias
    raise NotImplementedError  # TODO 2

In [ ]:
check("diagnose", lambda: diagnose(0.0, 0.02).startswith("good") and diagnose(0.0, 0.15).startswith("high variance") and diagnose(0.12, 0.14).startswith("high bias"),
      "(0, 0.02) → good enough; (0, 0.15) → high variance; (0.12, 0.14) → high bias.")
if ready("learning curve", "diagnose"):
    for name, lc in curves.items(): print(f"{name:<24} → {diagnose(lc[-1][1], lc[-1][2])}")
    check("curves tell different stories", lambda: diagnose(*curves["decision tree (depth 3)"][-1][1:]).startswith("high bias") and curves["RBF SVM"][-1][2] < curves["logistic regression"][-1][2],
          "The shallow tree should be diagnosed as high bias, and the SVM should end with the lowest validation error.")

<details><summary>▶ Solution</summary>

```python
def learning_curve(model, X_train, y_train, X_val, y_val, sizes, repeats=3, seed=0):
    rng = np.random.default_rng(seed)
    out = []
    for n in sizes:
        tr, va = [], []
        for _ in range(repeats):
            idx = rng.choice(len(X_train), n, replace=False)
            m = clone(model).fit(X_train[idx], y_train[idx])
            tr.append(1 - m.score(X_train[idx], y_train[idx])); va.append(1 - m.score(X_val, y_val))
        out.append((n, float(np.mean(tr)), float(np.mean(va))))
    return out
```

```python
def diagnose(train_err, val_err, target=0.03):
    gap = val_err - train_err
    if val_err <= target:
        return "good enough"
    if gap > 0.05:
        return "high variance: get more data or regularise"
    return "high bias: use a more flexible model or better features"
```
</details>

## Stretch goals
- Fit a power law `err ≈ a·n^(−b) + c` to the validation curve and extrapolate: how many examples would logistic regression need to reach 2 % error? (This is what "scaling laws" do for LLMs.)
- Add label noise (flip 10 % of labels). Which curves change, and how?
- Plot validation error vs. tree depth at n = 1000: the *other* axis of the bias–variance trade-off.